In [62]:
from src.improved_model import  BinarizingCNN
from src.utils import device
from settings import settings
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms
import torch

import numpy as np


cnn_model = BinarizingCNN()
cnn_model.load_state_dict(torch.load(settings.models_path / 'convnet_v2.pth'))
cnn_model.to(device)
cnn_model.eval_mode()


C:\Users\frrit\AppData\Local\Temp\ipykernel_17908\1866003032.py:12: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  cnn_model.load_state_dict(torch.load(settings.models_path /

In [25]:
# Define the number of classes in MNIST (digits 0-9)
num_classes = 10

# Define the transformation to apply to the images
transform = transforms.Compose([
    transforms.ToTensor(),  # Convert images to PyTorch tensors
])

# Custom transform to one-hot encode the labels
class OneHotEncode:
    def __init__(self, num_classes):
        self.num_classes = num_classes

    def __call__(self, label):
        return torch.eye(self.num_classes)[label]

# Load the full training dataset
full_train_dataset = datasets.MNIST(
    root=settings.data_path,
    train=True,
    download=True,
    transform=transform,
    target_transform=OneHotEncode(num_classes)
)

# Split the full training dataset into training and validation datasets
train_size = int(0.8 * len(full_train_dataset))  # 80% for training
val_size = len(full_train_dataset) - train_size  # 20% for validation
train_dataset, val_dataset = random_split(full_train_dataset, [train_size, val_size])

# Load the test dataset
test_dataset = datasets.MNIST(
    root=settings.data_path,
    train=False,
    download=True,
    transform=transform,
    target_transform=OneHotEncode(num_classes)
)

# Create DataLoaders for training, validation, and test sets
train_dataloader = DataLoader(train_dataset, batch_size=int(5e4), shuffle=True)
val_dataloader = DataLoader(val_dataset, batch_size=512, shuffle=False)
test_dataloader = DataLoader(test_dataset, batch_size=5000, shuffle=False)

In [56]:
train_data, _  = next(iter(train_dataloader))
train_data = train_data.to(device)



In [85]:

cnn_model = BinarizingCNN()
cnn_model.load_state_dict(torch.load(settings.models_path / 'convnet_v2.pth'))
cnn_model.to(device)
cnn_model.eval_mode()

old_bias = cnn_model.layer3.bias.clone()
old_output = cnn_model(train_data)

cnn_model.third_layer = lambda x: torch.where(cnn_model.layer3(x) >= 0, torch.tensor(1), torch.tensor(-1))
new_bias = torch.floor(old_bias.clone())
cnn_model.layer3.bias = torch.nn.Parameter(new_bias)
new_output = cnn_model(train_data)

assert  torch.all(old_output == new_output)

C:\Users\frrit\AppData\Local\Temp\ipykernel_17908\3103768316.py:2: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  cnn_model.load_state_dict(torch.load(settings.models_path / 

In [88]:
new_output.dtype

torch.int64

In [81]:
new_output[old_output != new_output]


tensor([], device='cuda:0')

In [31]:
# Check if all values in each column are the same
same_values_per_column = (x == x[0]).all(dim=0)

# Get the indices of columns where all values are the same
identical_columns = torch.nonzero(same_values_per_column, as_tuple=True)[0]

print(identical_columns)


tensor([ 0,  1,  2, 10, 11, 12, 14, 18, 19, 22, 26, 27, 28, 31, 44, 47, 50, 51,
        60, 70, 74], device='cuda:0')


In [46]:
print(torch.floor(old_bias))
old_bias

tensor([ 5.,  5., 11.,  9.,  5.,  6.,  7.,  9.,  9.,  7.], device='cuda:0',
       grad_fn=<FloorBackward0>)


tensor([ 5.8975,  5.8969, 11.0968,  9.2471,  5.2542,  6.4312,  7.2073,  9.1004,
         9.2056,  7.0819], device='cuda:0', grad_fn=<CloneBackward0>)